In [1]:
import os
import pandas as pd
import json
import random
from selenium.webdriver import ChromeOptions
import time
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver import Chrome
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys

option = ChromeOptions()
option.add_experimental_option('useAutomationExtension', False)
option.add_argument('blink-settings=imagesEnabled=false')
option.add_argument('--disable-blink-features=AutomationControlled')
option.add_argument('--disable-dev-shm-usage')
option.add_argument('--disable-gpu')
option.add_experimental_option('excludeSwitches', ['enable-automation'])
# option.add_argument('--headless')
option.add_argument('--no-sandbox')
option.add_argument('--proxy-server=http://f290.kdltps.com:15818')
option.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.0.0 Safari/537.36')
option.page_load_strategy = 'eager'


df_menu = pd.read_excel('C:/Users/YangTeng/Desktop/控制臂20231215-Rockauto信息补充.xlsx', header=0)
list_OE = df_menu['OE'].to_list()
list_OE

df = pd.DataFrame(columns=['Part_Number',
                           'Manufacturer',
                           'Vehicle',
                           'Note_1',
                           'Info',
                           'Note_2',
                           'Alternate_OE_Part_Numbers',
                           'Src',
                           'status'])

df

,Part_Number,Manufacturer,Vehicle,Note_1,Info,Note_2,Alternate_OE_Part_Numbers,Src,status


In [2]:
browser = Chrome(service=Service(r'C:\Users\YangTeng\Desktop\venv\chromedriver.exe'), options=option)

browser.maximize_window()

In [4]:
# for i in list_OE[333:400]:
#     print(i)
a = str('CMS76103')

browser.get('https://www.rockauto.com/en/partsearch/?partnum=UF510')

time.sleep(3)

wait = WebDriverWait(browser, 60)    

wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="partnum_partsearch_007"]')))

input_OE = browser.find_element(by=By.XPATH, value='//*[@id="partnum_partsearch_007"]')

input_OE.clear()

# a = str(int(Dorman[i]))

input_OE.send_keys(a,Keys.ENTER)

time.sleep(5)

url = browser.current_url

wait.until(EC.presence_of_element_located((By.XPATH, '//tbody[@class="listing-inner altrow-a-1  "]')))

list_part = browser.find_elements(by=By.XPATH, value='//tbody[@class="listing-inner altrow-a-1  "]') + browser.find_elements(by=By.XPATH, value='//tbody[@class="listing-inner altrow-a-0  "]')

list_part


[<selenium.webdriver.remote.webelement.WebElement (session="3c845b266e5c04255534c52859c7753a", element="370C9B4AE3AE46588CBFDD0E66124339_element_33")>,
 <selenium.webdriver.remote.webelement.WebElement (session="3c845b266e5c04255534c52859c7753a", element="370C9B4AE3AE46588CBFDD0E66124339_element_34")>,
 <selenium.webdriver.remote.webelement.WebElement (session="3c845b266e5c04255534c52859c7753a", element="370C9B4AE3AE46588CBFDD0E66124339_element_35")>,
 <selenium.webdriver.remote.webelement.WebElement (session="3c845b266e5c04255534c52859c7753a", element="370C9B4AE3AE46588CBFDD0E66124339_element_36")>,
 <selenium.webdriver.remote.webelement.WebElement (session="3c845b266e5c04255534c52859c7753a", element="370C9B4AE3AE46588CBFDD0E66124339_element_37")>]

In [5]:
list_part_number = [part.find_element(by=By.XPATH, value='./descendant::div[@class="listing-text-row-moreinfo-truck"]/span[@title="Buyer\'s Guide"]') for part in list_part]

list_part_number

part_index = [part_number.text for part_number in list_part_number].index(a)

part_index



2

In [6]:
manufacturer = list_part[part_index].find_element(by=By.XPATH, value='./descendant::div[@class="listing-text-row-moreinfo-truck"]/span[@class="listing-final-manufacturer "]').text

manufacturer



'ULTRA-POWER'

In [ ]:
list_part_number[part_index].click()
time.sleep(5)
list_vehicle = browser.find_element(by=By.XPATH, value='//div[@id="buyersguidepopup-outer_c"]').find_elements(by=By.XPATH, value='./descendant::table[@class="nobmp"]')
if len(list_vehicle) == 0:
    vehicle = ''
else:
    vehicle = list_vehicle[0].text

vehicle


In [ ]:
note_1 = ';'.join(row.text for row in list_part[part_index].find_elements(by=By.XPATH, value='./descendant::div[@class="listing-text-row-moreinfo-truck"]/span[@class="span-link-underline-remover"]'))

note_1

info = ';'.join(row.get_attribute('href') for row in list_part[part_index].find_elements(by=By.XPATH, value='./descendant::div[@class="listing-text-row-moreinfo-truck"]/a[@class="ra-btn ra-btn-moreinfo"]'))

info

note_2 = ';'.join(row.text for row in list_part[part_index].find_elements(by=By.XPATH, value='./descendant::div[@class="listing-text-row"]'))

note_2

oe = ';'.join(row.get_attribute('textContent') for row in list_part[part_index].find_elements(by=By.XPATH, value='./descendant::span[@title="Replaces these Alternate/ OE Part Numbers"]'))
oe = ';'.join(oe.split(', '))

oe

src = ';'.join(row.get_attribute('src') for row in list_part[part_index].find_elements(by=By.XPATH, value='./descendant::img[@class=" listing-inline-image"]'))

src

df_temp = pd.DataFrame([{'Part_Number': a,
                     'Manufacturer': manufacturer,
                     'Vehicle': vehicle,
                     'Note_1': note_1,
                     'Info': info,
                     'Note_2': note_2,
                     'Alternate_OE_Part_Numbers': oe,
                     'Src': src,
                     'url':url,
                     'status': 'ok'}])


df = pd.concat([df, df_temp], ignore_index=True).fillna('')
df

In [31]:
df.to_excel('C:/Users/YangTeng/Desktop/控制臂补充信息-7.xlsx', index=False)